# Contrastive Text Analysis with Dracor and Scattertext

> **Author:** Robert Jäschke (Humboldt-Universität zu Berlin, Germany), <a href="https://orcid.org/0000-0003-3271-9653"><img src="https://orcid.org/assets/vectors/orcid.logo.icon.svg" alt="ORCID iD" width="16" height="16" style="vertical-align: -2px"/> 0000-0003-3271-9653</a>  
> **Contributors:** Ingo Börner (University of Potsdam, Germany), <a href="https://orcid.org/0000-0001-8294-2541"><img src="https://orcid.org/assets/vectors/orcid.logo.icon.svg" alt="ORCID iD" width="16" height="16" style="vertical-align: -2px"/> 0000-0001-8294-2541</a>: update to PyDraCor 3 and the current DraCor API v1 (2026)  
> **Version:** 3 – PyDraCor 3, DraCor API v1 (November 2026); see the version history at the end of the notebook  
> **Status:** [![Status on dracor.org](https://img.shields.io/endpoint?url=https%3A%2F%2Fraw.githubusercontent.com%2Fdracor-org%2Fdracor-notebooks%2Fmain%2Fstatus%2Fbadges%2Fcontrastive-text-analysis.production.json)](https://dracor-org.github.io/dracor-notebooks/appendix/tested-environments.html) [![Status on staging](https://img.shields.io/endpoint?url=https%3A%2F%2Fraw.githubusercontent.com%2Fdracor-org%2Fdracor-notebooks%2Fmain%2Fstatus%2Fbadges%2Fcontrastive-text-analysis.staging.json)](https://dracor-org.github.io/dracor-notebooks/appendix/tested-environments.html)  
> **Tested with:** DraCor API 1.4.1 (production server) and 1.5.0-rc.1 (staging server), PyDraCor 3.0.2, Python 3.11, 2026-11-17

[Scattertext](https://github.com/JasonKessler/scattertext) enables the two-dimensional visualization of linguistic differences of two groups of text. We here use it to contrast the text of male and female speakers, based on the sex of the characters as encoded in DraCor.

## Requirements

We first install the libraries that are necessary to process the data, either with `pip install -r requirements.txt` (which includes the German language model of spaCy) or by uncommenting the lines below:

In [ ]:
#!pip install "pydracor>=3.0" scattertext six spacy spacy-transformers pandas
#!python -m spacy download de_dep_news_trf

## Acquiring the Corpus

We download the text of each character for [Goethe's Faust](https://dracor.org/ger/goethe-faust-eine-tragoedie) with [PyDraCor](https://pypi.org/project/pydracor/). The server is set in the variable `api_base`; to use another server, e.g. the staging server `https://staging.dracor.org/api/v1`, change it:

In [ ]:
%%time
from pydracor import DraCorAPI

api_base = "https://dracor.org/api/v1"
dracor = DraCorAPI(host=api_base)

play = dracor.get_play(corpus_name="ger", play_name="goethe-faust-eine-tragoedie")
text = play.get_spoken_text_by_character()

This gives us a list with information about all characters, including their sex and spoken text:

In [ ]:
text[23]

The later steps are easier if we transform this into tabular data with the columns *Speaker*, *Sex*, *Text*:

In [ ]:
import pandas as pd

table = [(c.label, c.sex, " ".join(c.text)) for c in text] # a list of tuples
df = pd.DataFrame(table, columns=["Speaker", "Sex", "Text"])  # a dataframe
df

What's the distribution of the speakers by sex?

In [ ]:
df.Sex.value_counts()

We remove texts from speakers whose sex is unknown to enable visualization in two dimensions:

In [ ]:
df = df[df.Sex != "UNKNOWN"]
df.Sex.value_counts()

## Building the Scattertext Page

We are basically following [this tutorial](https://github.com/JasonKessler/scattertext#using-scattertext-as-a-text-analysis-library-finding-characteristic-terms-and-their-associations). 

First, we load the trained language model: 

In [ ]:
import spacy
nlp = spacy.load("de_dep_news_trf")

Then we create a Scattertext corpus:

In [ ]:
import scattertext as st
corpus = st.CorpusFromPandas(df, category_col='Sex', text_col='Text', nlp=nlp).build()

And we print the terms "that differentiate the corpus from a general German corpus":

In [ ]:
list(corpus.get_scaled_f_scores_vs_background().index[:10])

Then we can create a HTML page showing the visualization of Scattertext:

In [ ]:
html = st.produce_scattertext_explorer(corpus,
          category='MALE',
          category_name='Male',
          not_category_name='Female',
          width_in_pixels=1000,
          metadata=df['Speaker'])
open(play.name + ".html", 'wb').write(html.encode('utf-8'))

Here's the result: [goethe-faust-eine-tragoedie.html](goethe-faust-eine-tragoedie.html)

## Use of AI

The update of this notebook to PyDraCor 3 and the current DraCor API v1 (version 3) was prepared with Claude Code (Anthropic) on behalf of the DraCor maintainers and reviewed by the author.

## Version history

The version history of this notebook is documented in [README.md](https://github.com/dracor-org/dracor-notebooks/blob/main/contrastive-text-analysis/README.md). In the [DraCor Notebooks book](https://dracor-org.github.io/dracor-notebooks/), it follows below.

```{include} README.md
:start-after: "## Version history"
```